---
**Copyright 2026 Nicola Vessio - Tutti i diritti riservati.**

Data di creazione dell'opera: 7 agosto 2026

Questo software (notebook, codice e relativa documentazione) e' opera di Nicola Vessio
ed e' protetto dalle norme vigenti in materia di diritto d'autore (L. 633/1941 e successive
modifiche, Convenzione di Berna, Direttiva 2009/24/CE sulla tutela giuridica dei programmi
per elaboratore).

Sono riservati all'autore tutti i diritti di utilizzazione economica dell'opera, inclusi
a titolo esemplificativo la riproduzione, la distribuzione, la modifica, l'adattamento,
la traduzione e la comunicazione al pubblico. Ogni uso non espressamente autorizzato
per iscritto dall'autore e' vietato.

Contatto: da definire.

---

# Report delle operazioni MT5 (da file) - Ausilio alla dichiarazione dei redditi

Il presente notebook legge un report di cronistoria esportato da MetaTrader 5
(in formato Excel .xlsx) ed elabora le operazioni in esso contenute, allo scopo di
produrre un report chiaro e ordinato a supporto della dichiarazione dei redditi.

A differenza della versione "live", questo modulo NON si collega al terminale MT5:
lavora su un file gia' esportato, quindi non richiede che l'applicazione sia aperta.
E' adatto a chi non puo' o non vuole tenere il terminale aperto, o deve elaborare
un estratto fornito da terzi.

## Come esportare il file da MetaTrader 5

Nel terminale MT5, nella scheda "Cronistoria" (Storico conto), fare clic destro e
scegliere "Report" -> "Foglio di calcolo XML di Office 2007". Questa voce genera un
file con estensione .xlsx, che e' il file da fornire a questo notebook.

Nota: viene accettato SOLO il formato .xlsx. Il formato "Report HTML" non e' supportato.

## Conti supportati

Lo strumento elabora il report di un conto MetaTrader 5, sia esso un conto operativo
diretto sia un conto in copytrading, presso qualsiasi broker.

## Funzionalita'

- Lettura del report .xlsx esportato da MT5
- Estrazione delle operazioni complete, gia' abbinate da MT5 (apertura + chiusura)
- Separazione tra operazioni di trading e movimenti di cassa (depositi/prelievi)
- Calcolo del profitto netto dei costi del broker (commissioni + swap)
- Produzione di un report Excel a quattro fogli

## Valuta del conto

Gli importi sono espressi nella valuta del conto (tipicamente USD). L'eventuale
conversione in euro ai fini della dichiarazione dei redditi italiana non e' gestita
qui ed e' a cura dello studio commercialistico.

## Prerequisiti

- Python 3.12 (ambiente Miniconda consigliato)
- Pacchetti: pandas, openpyxl
- Il file .xlsx esportato da MT5 (vedi istruzioni sopra)

## Nota importante

Questo notebook e' uno strumento di organizzazione ed esposizione dei dati, non una
consulenza fiscale. I dati vanno verificati e la dichiarazione e' gestita dal
commercialista o CAF. Il regime fiscale applicabile, le eventuali compensazioni, la
corretta imputazione di plusvalenze e minusvalenze, la conversione in euro, il quadro RW
e l'imposta di bollo sono di competenza dello studio commercialistico.

In [ ]:
# ============================================================
# IMPOSTAZIONI - Individuazione automatica del file da analizzare
# ============================================================
# Il notebook cerca automaticamente il report esportato da MT5 nella cartella
# in cui si trova il notebook stesso. Non occorre scrivere nomi o percorsi:
# e' sufficiente collocare il file .xlsx esportato da MT5 in questa cartella.
#
# IMPORTANTE: NON rinominare il file esportato da MT5. Deve mantenere il nome
# originale che MT5 gli assegna (inizia sempre con "ReportHistory"). Se il file
# viene rinominato, il notebook non riesce a individuarlo.
#
# Se nella cartella e' presente piu' di un report, viene usato il piu' recente.

import glob
import os

# Cerchiamo tutti i file che iniziano con "ReportHistory" ed hanno estensione .xlsx
file_trovati = glob.glob("ReportHistory*.xlsx")

if not file_trovati:
    raise SystemExit(
        "Nessun file trovato. Collocare il report .xlsx esportato da MT5 "
        "(nome originale tipo 'ReportHistory-XXXXXXXX.xlsx', da NON rinominare) "
        "nella cartella del notebook."
    )

# Se ce n'e' piu' d'uno, prendiamo il piu' recente (data di modifica piu' alta)
percorso_file = max(file_trovati, key=os.path.getmtime)

print(f"File individuato: {percorso_file}")
if len(file_trovati) > 1:
    print(f"(trovati {len(file_trovati)} file ReportHistory, usato il piu' recente)")

In [ ]:
# ============================================================
# Lettura del file e individuazione delle sezioni
# ============================================================
# Il report MT5 e' un unico foglio Excel diviso in sezioni, ciascuna introdotta
# da una riga-titolo: "Posizioni", "Ordini", "Affari".
# In questa cella carichiamo il foglio e individuiamo a quale riga inizia
# ciascuna sezione, cosi' le celle successive sapranno dove leggere i dati.
#   - "Posizioni" -> le operazioni gia' abbinate da MT5 (apertura + chiusura)
#   - "Affari"    -> da qui estrarremo in seguito solo i movimenti di cassa

from openpyxl import load_workbook

# Apriamo il file individuato nella cella precedente.
# data_only=True legge i valori (non le eventuali formule).
wb = load_workbook(percorso_file, data_only=True)
ws = wb.active   # il report MT5 ha un unico foglio

# Scorriamo la prima colonna alla ricerca delle righe-titolo di sezione.
# Salviamo il numero di riga in cui inizia ciascuna sezione.
riga_posizioni = None
riga_ordini    = None
riga_affari    = None

for riga in range(1, ws.max_row + 1):
    valore = ws.cell(row=riga, column=1).value
    if valore == "Posizioni":
        riga_posizioni = riga
    elif valore == "Ordini":
        riga_ordini = riga
    elif valore == "Affari":
        riga_affari = riga

# Controllo: se non troviamo le sezioni attese, il file non e' un report MT5 valido.
if riga_posizioni is None or riga_affari is None:
    raise SystemExit(
        "Il file non sembra un report di cronistoria MT5 valido "
        "(sezioni 'Posizioni'/'Affari' non trovate). Verificare di aver esportato "
        "il file corretto da MetaTrader 5."
    )

print("Sezioni individuate:")
print(f"  'Posizioni' inizia alla riga {riga_posizioni}")
print(f"  'Ordini'    inizia alla riga {riga_ordini}")
print(f"  'Affari'    inizia alla riga {riga_affari}")

## Nota sull'avviso "UserWarning" di openpyxl

Durante la lettura del file, Python potrebbe mostrare un avviso giallo simile a:

`UserWarning: Workbook contains no default style, apply openpyxl's default`

**Non e' un errore e non compromette in alcun modo i dati o il report.**

L'avviso segnala semplicemente che il file esportato da MetaTrader 5 non contiene
uno stile grafico predefinito (i report MT5 sono privi di formattazione). La libreria
di lettura ne applica automaticamente uno proprio e prosegue normalmente.

L'elaborazione e i calcoli non sono influenzati: l'avviso puo' essere ignorato.

In [ ]:
# ============================================================
# Estrazione delle operazioni (sezione "Posizioni")
# ============================================================
# La sezione "Posizioni" contiene le operazioni gia' abbinate da MT5:
# ogni riga e' un trade completo (apertura + chiusura) con il relativo profitto.
# Le colonne del report MT5 sono, nell'ordine:
#   Ora(apertura) | Posizione(id) | Simbolo | Tipo | Volume | Prezzo(apertura) |
#   S/L | T/P | Ora(chiusura) | Prezzo(chiusura) | Commissioni | Swap | Profitto
# Costruiamo un DataFrame con le stesse colonne usate dal modulo "live",
# cosi' il report Excel finale avra' struttura identica.

import pandas as pd
from datetime import datetime

# I dati delle Posizioni iniziano 2 righe dopo il titolo (titolo + intestazione),
# e finiscono 1 riga prima del titolo "Ordini".
prima_riga_dati = riga_posizioni + 2
ultima_riga_dati = riga_ordini - 1

trade_completi = []
for riga in range(prima_riga_dati, ultima_riga_dati + 1):
    valori = [ws.cell(row=riga, column=c).value for c in range(1, 14)]

    # Salta eventuali righe vuote
    if valori[0] is None:
        continue

    ora_apertura   = valori[0]    # datetime apertura
    position_id    = valori[1]    # id posizione
    simbolo        = valori[2]    # es. XAUUSD
    tipo           = valori[3]    # 'buy' o 'sell'
    volume         = valori[4]
    prezzo_apert   = valori[5]
    ora_chiusura   = valori[8]    # datetime chiusura
    prezzo_chius   = valori[9]
    commissione    = valori[10] if valori[10] is not None else 0.0
    swap           = valori[11] if valori[11] is not None else 0.0
    profitto       = valori[12] if valori[12] is not None else 0.0

    profit_netto = profitto + swap + commissione

    trade_completi.append({
        "position_id":   position_id,
        "symbol":        simbolo,
        "volume":        volume,
        "tipo":          str(tipo).upper(),                 # BUY / SELL
        "data_apertura": ora_apertura,
        "data_chiusura": ora_chiusura,
        "profit_lordo":  round(profitto, 2),
        "swap":          round(swap, 2),
        "commission":    round(commissione, 2),
        "profit_netto_costi_broker_lordo_imposte_USD":
            round(profit_netto, 2),
    })

df_trade = pd.DataFrame(trade_completi).sort_values("data_chiusura").reset_index(drop=True)

print(f"Operazioni estratte: {len(df_trade)}")
print(f"Profitto netto totale (USD): {df_trade['profit_netto_costi_broker_lordo_imposte_USD'].sum():.2f}")
print()
print(df_trade.head(12))

In [ ]:
# ============================================================
# Estrazione dei movimenti di cassa (dalla sezione "Affari")
# ============================================================
# I depositi e prelievi non compaiono tra le "Posizioni" (che sono solo trade):
# si trovano nella sezione "Affari", nelle righe il cui TIPO e' "balance".
# Queste righe rappresentano movimenti di denaro (versamenti, prelievi, aggiustamenti),
# NON operazioni di trading, e vanno tenute separate.
#
# Struttura colonne sezione "Affari":
#   Ora | Affare(id) | Simbolo | Tipo | Direzione | Volume | Prezzo | Ordine |
#   Commissioni | Spese | Swap | Profitto | Bilancio | Commento
# Nota: per i movimenti di cassa la colonna "Simbolo" e' vuota e la parola
# "balance" si trova nella colonna "Tipo" (colonna 4).
# L'importo del movimento e' nella colonna "Profitto" (colonna 12).

# I dati degli Affari iniziano 2 righe dopo il titolo (titolo + intestazione)
prima_riga_affari = riga_affari + 2

movimenti_cassa = []
for riga in range(prima_riga_affari, ws.max_row + 1):
    tipo = ws.cell(row=riga, column=4).value   # colonna "Tipo"

    # Ci fermiamo quando finiscono i dati: se anche l'ora (col 1) e' vuota, usciamo
    if ws.cell(row=riga, column=1).value is None:
        break

    # Ci interessano SOLO le righe di tipo "balance" (movimenti di cassa)
    if tipo is not None and str(tipo).lower() == "balance":
        ora      = ws.cell(row=riga, column=1).value    # data/ora del movimento
        affare   = ws.cell(row=riga, column=2).value    # id del movimento
        importo  = ws.cell(row=riga, column=12).value   # colonna "Profitto" = importo
        commento = ws.cell(row=riga, column=14).value   # descrizione (es. "Withdrawal Via NAB")

        importo = importo if importo is not None else 0.0
        movimenti_cassa.append({
            "id":       affare,
            "data":     ora,
            "importo":  round(importo, 2),
            "tipo":     "Deposito" if importo > 0 else "Prelievo",
            "commento": commento if commento is not None else "",
        })

print(f"Movimenti di cassa trovati: {len(movimenti_cassa)}")
for m in movimenti_cassa:
    print(f"  {m['data']} | {m['tipo']} | {m['importo']:.2f} USD | {m['commento']}")

In [ ]:
# ============================================================
# Lettura dei dati del conto dall'intestazione del file
# ============================================================
# Nel modulo "live" i dati del conto arrivavano da account_info().
# Qui invece li leggiamo dall'intestazione del report (righe 2-5), dove MT5
# scrive nome, conto, societa' e data. Il numero conto sta in una riga unica
# insieme a valuta, server, tipo e modalita' conto, quindi la spacchettiamo
# nei suoi singoli campi.
#
# Esempio riga "Conto:"  ->  "12345678 (USD, BrokerServer-Live, real, Hedge)"
#   numero conto = 12345678
#   valuta       = USD
#   server       = BrokerServer-Live
#   tipo conto   = real
#   modalita'    = Hedge

# Leggiamo i valori grezzi dall'intestazione (colonna 4)
nome_intestatario = ws.cell(row=2, column=4).value or "n/d"
riga_conto        = ws.cell(row=3, column=4).value or ""
societa           = ws.cell(row=4, column=4).value or "n/d"

# Spacchettiamo la riga del conto.
# Formato: "NUMERO (VALUTA, SERVER, TIPO, MODALITA)"
numero_conto = "n/d"
valuta       = "n/d"
server       = "n/d"
tipo_conto   = "n/d"
modalita     = "n/d"

if riga_conto:
    # La parte prima della parentesi e' il numero conto
    if "(" in riga_conto:
        numero_conto = riga_conto.split("(")[0].strip()
        # La parte tra parentesi contiene valuta, server, tipo, modalita' (separati da virgola)
        dentro_parentesi = riga_conto.split("(")[1].rstrip(")")
        parti = [p.strip() for p in dentro_parentesi.split(",")]
        if len(parti) >= 1:
            valuta = parti[0]        # 1o = valuta (es. USD)
        if len(parti) >= 2:
            server = parti[1]        # 2o = server
        if len(parti) >= 3:
            tipo_conto = parti[2]    # 3o = tipo conto (es. real)
        if len(parti) >= 4:
            modalita = parti[3]      # 4o = modalita' (es. Hedge)
    else:
        numero_conto = riga_conto.strip()

# Raccogliamo tutto in un dizionario, come "info" del modulo live
info_conto = {
    "intestatario": nome_intestatario,
    "societa":      societa,
    "numero_conto": numero_conto,
    "server":       server,
    "valuta":       valuta,
    "tipo_conto":   tipo_conto,
    "modalita":     modalita,
}

print("Dati del conto letti dal file:")
for chiave, valore in info_conto.items():
    print(f"  {chiave}: {valore}")